# WandB Experiment Analysis

Generates LaTeX tables:
1. **Main table** - 10 key models at end checkpoint (with AVG rows)
2. **Other runs table** - Remaining runs (same format, abbreviated)

In [1]:
import wandb
import pandas as pd
import numpy as np
from wandb_helper import (
    build_metrics_table,
    to_latex_transposed,
    to_latex_other_runs,
    add_display_names,
    print_available_checkpoints,
    fetch_run_history,
)
from mapping import EXPERIMENT_MANIFEST

%load_ext autoreload
%autoreload 2

## 1. Configuration

In [2]:
ENTITY = "cyhsm"
PROJECT = "loom"

CHECKPOINTS = {
    "early": 5000,
    "mid": 20000,
    "end": 35000,
}

CE_AVG_WINDOW = 100

## 2. Experiment Manifest & Short Names

In [3]:
SHORT_NAMES = {
    # Main table models
    "baseline_buckets_mvd": "Base",
    "loop3_buckets_mvd": "L3",
    "loop5_buckets_mvd": "L5",
    "loop7_buckets_mvd": "L7",
    "loop3_iso_mvd": r"L3\textsubscript{IF}",
    "baseline_isoparam_to_L1024G512memory": r"M-B\textsubscript{IP}",
    "loop3_L1024G512_individualMemory_init-3": r"M\textsubscript{-3}",
    "loop3_L1024G512_individualMemory_init0": r"M\textsubscript{0}",
    "loop3_L1024G512_individualMemory_init3": r"M\textsubscript{3}",
    "baseline_isoflop_to_L1024G512memory_and_3loops": r"M-B\textsubscript{IF}",
    
    # Other runs (not in main table)
    "loop1_L512G1024": r"L1\textsubscript{512/1k}",
    "loop1_L512G4096": r"L1\textsubscript{512/4k}",
    "loop1_L1024G512": r"L1\textsubscript{1k/512}",
    "loop1_L4096G512": r"L1\textsubscript{4k/512}",
    "loop3_L1024G512_cyclical": r"L3\textsubscript{cyc}",
    "loop3_L1024G512_ponder001": r"L3\textsubscript{p+}",
    "loop3_L1024G512_ponder-001": r"L3\textsubscript{p-}",
    "loop3_L1024G512_individualMemory_frozenmem": r"M\textsubscript{frz}",
    "loop5_L1024G512": r"L5\textsubscript{1k}",
    "loop5_iso_mvd": r"L5\textsubscript{IF}",
    "loop9_buckets": "L9",
}

## 3. Main Table Models (10 for paper)

In [4]:
MAIN_TABLE_MODELS = [
    "baseline_buckets_mvd",
    "loop3_buckets_mvd",
    "loop5_buckets_mvd",
    "loop7_buckets_mvd",
    "loop3_iso_mvd",
    "baseline_isoparam_to_L1024G512memory",
    "loop3_L1024G512_individualMemory_init-3",
    "loop3_L1024G512_individualMemory_init0",
    "loop3_L1024G512_individualMemory_init3",
    "baseline_isoflop_to_L1024G512memory_and_3loops",
]

## 4. Debug (Optional)

In [5]:
# Uncomment to inspect a run:
# print_available_checkpoints(ENTITY, PROJECT, "dx7k84zu")

## 5. Fetch All Data

In [6]:
experiments = [{**exp, "display_name": exp["name"]} for exp in EXPERIMENT_MANIFEST]
print(f"Fetching {len(experiments)} experiments...")

Fetching 21 experiments...


In [ ]:
df_all = build_metrics_table(
    entity=ENTITY,
    project=PROJECT,
    experiments=experiments,
    checkpoints=CHECKPOINTS,
    ce_avg_window=CE_AVG_WINDOW,
)
print(f"\nFetched data: {df_all.shape}")

Fetching: baseline_buckets_mvd (dx7k84zu)...
  Loading history for baseline_buckets_mvd (dx7k84zu), state: finished
  Loaded 38642 rows, 48 columns
Fetching: baseline_isoflop_to_L1024G512memory_and_3loops (acw5j6ox)...
  Loading history for baseline_isoflop_to_L1024G512memory_and_3loops (acw5j6ox), state: finished
  Loaded 38628 rows, 47 columns
Fetching: baseline_isoparam_to_L1024G512memory (zxwh48wu)...
  Loading history for baseline_isoparam_to_L1024G512memory (zxwh48wu), state: finished
  Loaded 38621 rows, 47 columns
Fetching: loop1_L512G1024 (dy705ftt)...
  Loading history for loop1_L512G1024 (dy705ftt), state: finished
  Loaded 38635 rows, 125 columns
Fetching: loop1_L512G4096 (1ket85g0)...
  Loading history for loop1_L512G4096 (1ket85g0), state: finished
  Loaded 38635 rows, 125 columns
Fetching: loop1_L1024G512 (2h4fgihj)...
  Loading history for loop1_L1024G512 (2h4fgihj), state: finished
  Loaded 38642 rows, 125 columns
Fetching: loop1_L4096G512 (jhtbyhil)...
  Loading histo

In [ ]:
# Quick preview
df_all[["Model"] + [("end", "CS Acc"), ("end", "CS BPB"), ("end", "Math BPB")]].round(4)

---
## 6. MAIN TABLE: 10 Models (with AVG rows)

In [ ]:
latex_main = to_latex_transposed(
    df=df_all,
    checkpoint="end",
    model_order=MAIN_TABLE_MODELS,
    short_names=SHORT_NAMES,
    caption="Comparison of main models. \\textbf{Bold} = best result.",
    label="tab:main_results",
    bold_best=True,
    decimals=4,
)
print(latex_main)

In [ ]:
with open("./tables/table_main_end.tex", "w") as f:
    f.write(latex_main)
print("Saved ./tables/table_main_end.tex")

---
## 7. OTHER RUNS: Remaining experiments (same format)

In [ ]:
# END checkpoint
latex_other_end = to_latex_other_runs(
    df=df_all,
    checkpoint="end",
    exclude_models=MAIN_TABLE_MODELS,
    short_names=SHORT_NAMES,
    caption="Other runs - End (step 35000). \\textbf{Bold} = best.",
    label="tab:other_end",
    bold_best=True,
    decimals=4,
)
print("=" * 60)
print("OTHER RUNS - END")
print("=" * 60)
print(latex_other_end)

In [ ]:
# MID checkpoint
latex_other_mid = to_latex_other_runs(
    df=df_all,
    checkpoint="mid",
    exclude_models=MAIN_TABLE_MODELS,
    short_names=SHORT_NAMES,
    caption="Other runs - Mid (step 20000). \\textbf{Bold} = best.",
    label="tab:other_mid",
    bold_best=True,
    decimals=4,
)
print("=" * 60)
print("OTHER RUNS - MID")
print("=" * 60)
print(latex_other_mid)

In [ ]:
# EARLY checkpoint
latex_other_early = to_latex_other_runs(
    df=df_all,
    checkpoint="early",
    exclude_models=MAIN_TABLE_MODELS,
    short_names=SHORT_NAMES,
    caption="Other runs - Early (step 5000). \\textbf{Bold} = best.",
    label="tab:other_early",
    bold_best=True,
    decimals=4,
)
print("=" * 60)
print("OTHER RUNS - EARLY")
print("=" * 60)
print(latex_other_early)

In [ ]:
# Save all
with open("./tables/table_other_end.tex", "w") as f:
    f.write(latex_other_end)
with open("./tables/table_other_mid.tex", "w") as f:
    f.write(latex_other_mid)
with open("./tables/table_other_early.tex", "w") as f:
    f.write(latex_other_early)
print("Saved: ./tables/table_other_end.tex, ./tables/table_other_mid.tex, ./tables/table_other_early.tex")

---
## 8. MAIN TABLE at all checkpoints (for reference)

In [ ]:
# Main table at MID
latex_main_mid = to_latex_transposed(
    df=df_all,
    checkpoint="mid",
    model_order=MAIN_TABLE_MODELS,
    short_names=SHORT_NAMES,
    caption="Main models - Mid (step 20000).",
    label="tab:main_mid",
    bold_best=True,
    decimals=4,
)
print("MAIN TABLE - MID")
print(latex_main_mid)

In [ ]:
# Main table at EARLY
latex_main_early = to_latex_transposed(
    df=df_all,
    checkpoint="early",
    model_order=MAIN_TABLE_MODELS,
    short_names=SHORT_NAMES,
    caption="Main models - Early (step 5000).",
    label="tab:main_early",
    bold_best=True,
    decimals=4,
)
print("MAIN TABLE - EARLY")
print(latex_main_early)

---
## 9. Combined Output

In [ ]:
combined = f"""
% ============================================================
% MAIN TABLE - END (step 35000)
% ============================================================
{latex_main}

% ============================================================
% MAIN TABLE - MID (step 20000)
% ============================================================
{latex_main_mid}

% ============================================================
% MAIN TABLE - EARLY (step 5000)
% ============================================================
{latex_main_early}

% ============================================================
% OTHER RUNS - END
% ============================================================
{latex_other_end}

% ============================================================
% OTHER RUNS - MID
% ============================================================
{latex_other_mid}

% ============================================================
% OTHER RUNS - EARLY
% ============================================================
{latex_other_early}
"""

with open("./tables/all_tables.tex", "w") as f:
    f.write(combined)
print("Saved: ./tables/all_tables.tex")

---
## 10. Raw CSV Export

In [ ]:
df_all.to_csv("all_metrics.csv", index=False)
print("Saved: all_metrics.csv")